# ❄️ Snowflake Unstructured to Insights: Hands-On Lab ❄️

In this session, you will learn how to:
- Utilize Snowflake's Cortex AI functions to classify and parse documents
- Extract key information from documents using Gen AI 
- Implement and LLM-as-a-judge with Snowflake Cortex
- Automate a processing pipeline with Snowflake streams, tasks, and stored procedures

### What are Cortex AI Functions

Task-specific functions for common problems that require little customization. For example, text summaries, translations, PII redaction, etc. 

- SQL‑first
- Secure and governed
- Scalable within existing Snowflake workloads

### An example using AI_FILTER for fuzzy-matching

Historically, most fuzzy-matching has involved significant text cleaning and normalization followed by the calculation of some sort of edit distance metric. The simplest approach has been calculating the number of single‑character edits needed to transform one string into another (Levenshtein distance). 

```
kitten
sitting
```

| Step | Operation | Result |
|-----:|-----------|--------|
| 1 | Substitute `k` → `s` | `sitten` |
| 2 | Substitute `e` → `i` | `sittin` |
| 3 | Insert `g` | `sitting` |

---

#### Edit Count

- Substitutions: **2**
- Insertions: **1**
- Deletions: **0**

---

In addition to an edit distance, Snowflake has also offered a [Jaro-Winkler Similarity](https://en.wikipedia.org/wiki/Jaro%E2%80%93Winkler_distance) calculation. A Jaro similarity takes matching characters and transpositions into account, but normalizes for the word length. For example, FAREMVIEL and FARMVILLE will end up with a similarity of 88. When you add the Winkler part of Jaro-Winkler, you reward shared prefixes. So in our FAREMVIEL and FARMVILLE example, because they both start with 'FAR', the score gets bumped to 91. 

---
#### Example Cells Below
1. Creates a two column table. Some of the items are equivalent, but with different spellings. Other items are very different, but with similar spellings. 
2. Shows the result using traditional fuzzy matching approaches.
3. Shows results using generative AI with AI_FILTER. 

We'll pause briefly to run the cells and let you edit the DDL with any examples of your choosing. 



In [ ]:
CREATE OR REPLACE TABLE similar_strings (
    thing1 STRING,
    thing2 STRING
);

INSERT INTO similar_strings (thing1, thing2)
VALUES
    ('Springfield, MN', 'Springfield, MO'),
    ('Newark', 'Newburgh'),
    ('CornMaster Pro', 'CornMaster Plus'),
    ('Minneapolis–Saint Paul', 'Twin Cities'),
    ('William Johnson', 'Bill Johnson'),
    ('Minnesota Mining and Manufacturing', '3M'),
    ('Michael Brown', 'Michelle Brown');


In [ ]:
SELECT *,
JAROWINKLER_SIMILARITY(THING1, THING2) AS STRING_SIMILARITY,
EDITDISTANCE(THING1, THING2) AS LEVENSHTEIN_DISTANCE
FROM similar_strings
ORDER BY STRING_SIMILARITY DESC;

In [ ]:

SELECT * 
FROM similar_strings 
WHERE AI_FILTER(
        PROMPT('The two items, names, or places refer to the same thing: {0}, {1}',
        THING1, 
        THING2
        )
    );

In the cells below, we implement examples of sentiment and translation. Use your imagination and fill in the cells to test the functions. If you're feeling unimaginative, copy-paste the example from the subsequent markdown cell. 

In [ ]:
SELECT AI_SENTIMENT('Enter some text here');

```sql
SELECT AI_SENTIMENT(
'What you\'ve just said is one of the most insanely idiotic things I have ever heard. At no point in your rambling, incoherent response were you even close to anything that could be considered a rational thought. Everyone in this room is now dumber for having listened to it. I award you no points, and may God have mercy on your soul.'
);
```

Getting the overall sentiment of a text is pretty vanilla, and in some cases not all that useful. Imagine a situation where you want sentiment associated with certain topics, for example the quality of your product or the pricing of your service. 

Try adding categories/entities/aspects of the text that you'd like sentiment on. For more information, see the [documentation](https://docs.snowflake.com/en/sql-reference/functions/ai_sentiment)

In [ ]:
SELECT AI_SENTIMENT(
'Enter some text',
['category1', 'category2']
);

```sql
SELECT AI_SENTIMENT(
'Hawaii...alright, that\'s good. Hard to trace I guess. Wait, you changed your name to McLovin? McLovin!!! What kind of a stupid name is that Fogel? Are you trying to be an Irish R&B singer.',
['location selection', 'name selection']
);
```

In [ ]:
SELECT AI_TRANSLATE(
'Use your imagination',
'current_language', -- https://docs.snowflake.com/en/sql-reference/functions/ai_translate#usage-notes
'desired_language'
);

```sql
SELECT AI_TRANSLATE(
'
ドニーは腕のいいボウラーで、そして立派な男でした。彼は我々の仲間でした。彼はアウトドアを愛した男で……そしてボウリングを愛し、またサーファーとして、ラホヤからレオ・カリージョ、そして……ピズモまで、南カリフォルニアの海岸を巡っていました。
彼は、その世代の多くの若者たちと同じように、あまりにも早くこの世を去りました。主よ、あなたの御心のままに、あなたは彼をお取りになりました。ケサン、ランドック、ヒル364で命を落とした、あまたの才能あふれる若者たちをお取りになったように。
彼らは命を捧げました。そしてドニーも、同じように命を捧げたでしょう。ボウリングを愛したドニー。

そこで、セオドア・ドナルド・カラボツォスよ。あなたの最期の願いがきっとそうであったであろうという我々の思いに従い、あなたのこの世の遺骸を、あなたがこよなく愛した太平洋の懐へとお返しします。

おやすみなさい、愛しき王子よ。
',
'ja',
'en'
);
```

### Document Pipeline

#### Problem Statement

Let's imagine we own some hypothetical business that contracts with with outside vendors to provide goods and services. Because our business is relatively immature, PDFs are getting dumped on us and someone manually sorts out invoices and contracts and then reviews contract terms. 

#### Approach

We'll build a multi-step pipeline that extracts text from PDFs, sorts them into contracts vs. invoices, completes a preliminary review of contracts, then flags those needing additional human review. 

To break this down into bite-sized pieces, in the before-lunch portion of this lab we won't talk about automating anything. We'll create a number of tables to view our intermediate results. 

To work with unstructured data, Snowflake has provided a [`FILE` data type](https://docs.snowflake.com/en/sql-reference/data-types-unstructured#label-data-types-file). In the query below, we'll reference a stage where PDFs have been preloaded for you. We'll convert to a`FILE` data type using the TO_FILE function. 

In [ ]:

CREATE OR REPLACE TABLE PDF_LINKS AS (
SELECT RELATIVE_PATH AS DOC_LOCATION,
       TO_FILE('@pdf_dump', RELATIVE_PATH) AS DOC_FILE 
FROM DIRECTORY(@pdf_dump));

In [ ]:
SELECT * FROM PDF_LINKS;

We'll now use Snowflake's [AI_PARSE_DOCUMENT](https://docs.snowflake.com/en/sql-reference/functions/ai_parse_document) function to pull the text out of the PDFs. 

The function provides two methods for extraction. 

1. **OCR** mode. This will extract just the text from the document. If you've ever used Tesseract with its default settings, this provides a similar experience. 
2. **Layout** mode. This will utilize markdown to try to preserve things like table structure. It is more similar to Landing.AI or using Unstructured's LayoutParser. It will be slower than OCR, but for many tasks the layout may improve accuracy when using Gen AI. 

In [ ]:
CREATE OR REPLACE TABLE PDF_CONTENT AS (
SELECT DOC_LOCATION, 
       TO_VARCHAR(
            AI_PARSE_DOCUMENT(
                DOC_FILE, 
                {'mode': 'LAYOUT'}
                )
            ) AS DOCUMENT_MARKDOWN 
FROM PDF_LINKS
);

In [ ]:
SELECT * FROM PDF_CONTENT;

In the next cell we'll use another function called [AI_CLASSIFY](https://docs.snowflake.com/en/sql-reference/functions/ai_classify). We'll provide no examples (zero-shot prompting). In a future step we'll get more complex. 

`AI_CLASSIFY` will return an `OBJECT` data type (think JSON). The syntax `:labels[0]::VARCHAR` is simply extracting the first item from the list associated with 'labels' key and casting it as a `VARCHAR`. 

In [ ]:
CREATE OR REPLACE TABLE DOCUMENT_CLASSIFICATION AS (
SELECT *, 
AI_CLASSIFY(DOCUMENT_MARKDOWN, ['Invoice', 'Contract', 'Other']):labels[0]::VARCHAR AS DOC_TYPE
FROM PDF_CONTENT
);

In [ ]:
SELECT DOC_LOCATION, 
       DOC_TYPE
FROM DOCUMENT_CLASSIFICATION;

In the below cell we'll use a function called [AI_EXTRACT](https://docs.snowflake.com/en/sql-reference/functions/ai_extract) in addition to AI_CLASSIFY to pull information from the contracts in a way similar to what a human might do. 

AI_EXTRACT is a zero-shot extraction from either text or a file. If you click into the documentation, you'll see four different ways to specify the `responseFormat`. In our example, I've just used the simple object schema syntax. 

```sql
AI_EXTRACT(DOCUMENT_MARKDOWN, {
    'VENDOR_NAME': 'What is the name of the vendor providing the good or service?', 
    'CONTRACT_DATE': 'What is the date of the contract?',
    'TERMINATION_LANGUAGE': 'How do the parties involved in this agreement terminate it if necessary.',
    'LATE_PAYMENT_PENALTY': 'What happens if a payment is not made on-time?',
    'DISCOUNT_LANGUAGE': 'Describe any discounts available for early payment.'
    }
) AS FIELD_EXTRACTION
```
For this example, we're asking `AI_CLASSIFY` to do a much more complicated task than in the prior invoice/contract/other example. If you clicked through to the documentation earlier, you may have seen that label descriptions and a  `config_object` were optional. Here we're using them both to try to achieve better results. The config_object contains a task description and several examples, known as few-shot prompting. 

```sql
AI_CLASSIFY(
  DOCUMENT_MARKDOWN,
  [
    {'label': 'NET_10', 'description': 'payment due 10 days after invoice'},
    {'label': 'NET_15', 'description': 'payment due 15 days after invoice'},
    {'label': 'NET_20', 'description': 'payment due 20 days after invoice'},
    {'label': 'NET_21', 'description': 'payment due 20 days after invoice'},
    {'label': 'NET_30', 'description': 'payment due 30 days after invoice'},
    {'label': 'NET_45', 'description': 'payment due 45 days after invoice'},
    {'label': 'NET_60', 'description': 'payment due 60 days after invoice'},
    {'label': 'NET_90', 'description': 'payment due 90 days after invoice'},
    {'label': 'DUE_ON_RECEIPT', 'description': 'payment due immediately upon receipt'},
    {'label': 'ADVANCED_PAYMENT', 'description': 'payment required before goods or services are delivered'},
    {'label': 'MILESTONE_PAYMENT', 'description': 'payment due upon completion of project milestones'},
    {'label': 'MONTHLY_IN_ARREARS', 'description': 'payment due on the first or last day of the month for services previously provided'}
  ],
  {
    'task_description': 'Identify the payment terms described in the contract',
    'output_mode': 'single',
    'examples': [
      {
        'input': 'Invoice payable within ten days of receipt.',
        'labels': ['NET_10'],
        'explanation': 'the text specifies payment within ten days'
      },
      {
        'input': 'Prepayment is required at the start of each period.',
        'labels': ['ADVANCED_PAYMENT'],
        'explanation': 'payment must be made prior to service delivery'
      },
      {
        'input': 'Payment is due immediately upon receipt of invoice.',
        'labels': ['DUE_ON_RECEIPT'],
        'explanation': 'the text states payment is due immediately'
      },
      {
        'input': 'Payments will be made upon completion of each project phase.',
        'labels': ['MILESTONE_PAYMENT'],
        'explanation': 'payment is tied to milestones'
      },
      {
        'input': 'Balance payable at the end of each month.',
        'labels': ['MONTHLY_IN_ARREARS'],
        'explanation': 'payment is due at month end'
      }
    ]
  }
) AS PAYMENT_TERMS
```



In [ ]:
CREATE OR REPLACE TABLE CONTRACT_EXTRACTION AS (
SELECT *, 
AI_EXTRACT(DOCUMENT_MARKDOWN, {
    'VENDOR_NAME': 'What is the name of the vendor providing the good or service?', 
    'CONTRACT_DATE': 'What is the date of the contract?',
    'TERMINATION_LANGUAGE': 'How do the parties involved in this agreement terminate it if necessary.',
    'LATE_PAYMENT_PENALTY': 'What happens if a payment is not made on-time?',
    'DISCOUNT_LANGUAGE': 'Describe any discounts available for early payment.'
    }
) AS FIELD_EXTRACTION,
AI_CLASSIFY(
  DOCUMENT_MARKDOWN,
  [
    {'label': 'NET_10', 'description': 'payment due 10 days after invoice'},
    {'label': 'NET_15', 'description': 'payment due 15 days after invoice'},
    {'label': 'NET_20', 'description': 'payment due 20 days after invoice'},
    {'label': 'NET_21', 'description': 'payment due 20 days after invoice'},
    {'label': 'NET_30', 'description': 'payment due 30 days after invoice'},
    {'label': 'NET_45', 'description': 'payment due 45 days after invoice'},
    {'label': 'NET_60', 'description': 'payment due 60 days after invoice'},
    {'label': 'NET_90', 'description': 'payment due 90 days after invoice'},
    {'label': 'DUE_ON_RECEIPT', 'description': 'payment due immediately upon receipt'},
    {'label': 'ADVANCED_PAYMENT', 'description': 'payment required before goods or services are delivered'},
    {'label': 'MILESTONE_PAYMENT', 'description': 'payment due upon completion of project milestones'},
    {'label': 'MONTHLY_IN_ARREARS', 'description': 'payment due on the first or last day of the month for services previously provided'}
  ],
  {
    'task_description': 'Identify the payment terms described in the contract',
    'output_mode': 'single',
    'examples': [
      {
        'input': 'Invoice payable within ten days of receipt.',
        'labels': ['NET_10'],
        'explanation': 'the text specifies payment within ten days'
      },
      {
        'input': 'Prepayment is required at the start of each period.',
        'labels': ['ADVANCED_PAYMENT'],
        'explanation': 'payment must be made prior to service delivery'
      },
      {
        'input': 'Payment is due immediately upon receipt of invoice.',
        'labels': ['DUE_ON_RECEIPT'],
        'explanation': 'the text states payment is due immediately'
      },
      {
        'input': 'Payments will be made upon completion of each project phase.',
        'labels': ['MILESTONE_PAYMENT'],
        'explanation': 'payment is tied to milestones'
      },
      {
        'input': 'Balance payable at the end of each month.',
        'labels': ['MONTHLY_IN_ARREARS'],
        'explanation': 'payment is due at month end'
      }
    ]
  }
) AS PAYMENT_TERMS
FROM DOCUMENT_CLASSIFICATION
WHERE DOC_TYPE = 'Contract'
);


In [ ]:
SELECT * FROM CONTRACT_EXTRACTION;

Not every task is going to fit nicely into the box of one of the pre-baked AISQL functions. The [AI_COMPLETE](https://docs.snowflake.com/en/sql-reference/functions/ai_complete) function allows you to fully control your prompt, context that is passed into it, and your output structure. It works similarly to the chat_completions API endpoint provided by OpenAI or text_completions provided by Anthropic. 

Below we'll use `AI_COMPLETE` to implement an "llm-as-a-judge" step. The concept of “LLM-as-a-judge” refers to using a large language model to evaluate, score, or compare the outputs of other models (or prompts). It is commonly used in model evaluation, prompt testing, and quality assurance to automate assessments that would otherwise require human reviewers. 

In [ ]:
SELECT
  *,
  AI_COMPLETE(
    model => 'claude-4-sonnet',
    prompt => CONCAT(
    'You are reviewing a contract analysis for accuracy and completeness.

    Review the extracted contract fields and classified payment terms.

    Flag this contract for human review if:
    - Any required field is missing or incorrect
    - The required fields to check for correctness are 
        1. Vendor Name
        2. Termination Language
        3. Late Payment Penalty
    - The payment terms do not match the contract language
    - Payment terms need to only fit a category.
    - The available payment categories are:
        NET_10,
        NET_15, 
        NET_20,
        NET_21, 
        NET_30, 
        NET_45, 
        NET_60, 
        NET_90, 
        DUE_ON_RECEIPT, 
        ADVANCED_PAYMENT,
        MILESTONE_PAYMENT,
        MONTHLY_IN_ARREARS
    - Payment terms need not contain a full explanation of remitting payment, discounts, etc.
    - The vendor does not match the person or company providing the goods or services
    - The contract language is ambiguous or unusual
    - You are not judging output formatting
    - There is conflicting or unclear payment or termination language',
    '<document_text>', DOCUMENT_MARKDOWN, '</document_text>',
    '<extracted_fields>', FIELD_EXTRACTION::VARCHAR, '</extracted_fields>',
    '<payment_terms>', PAYMENT_TERMS::VARCHAR, '</payment_terms>',

    'Stop and review your work. Make sure you are following instructions exactly.'
    ),
    response_format => {
        'type': 'json',
        'schema': {
            'type': 'object',
            'properties': {
                'needs_human_review': {
                    'type': 'boolean'
                },
                'issues': {
                    'type': 'string',
                    'description': 'a list of identified issues (empty if none)'
                },
                'summary': {
                    'type': 'string',
                    'description': 'a short explanation (empty if no issues)'
                }         
            }
        }
    }
  ) AS LLM_JUDGE_RESULT
FROM CONTRACT_EXTRACTION;

### Lunch Time

Get some food.

In [ ]:
SET COURSE = 'One Ring to rule them all, One Ring to find them, One Ring to bring them all and in the darkness bind them.';

SET PIN = '["rule","find","bring","bind"]';
SET PROMPT = 'file me in';
SELECT ai_extract($COURSE, {'label': $prompt}):response['label']::varchar AS MY_ANSWER,
CASE WHEN UPPER(MY_ANSWER) = UPPER($PIN) THEN TRUE ELSE FALSE END AS IN_THE_HOLE,
LENGTH($PROMPT) AS STROKES;

In [ ]:
SET COURSE = 'Try not. Do. Or do not. There is no try.';
SET PIN = 'try';
SET PROMPT = 'fill me in';
SELECT ai_extract($COURSE, {'label': $prompt}):response['label']::varchar AS MY_ANSWER,
CASE WHEN UPPER(MY_ANSWER) = UPPER($PIN) THEN TRUE ELSE FALSE END AS IN_THE_HOLE,
LENGTH($PROMPT) AS STROKES;



In [ ]:
SET COURSE = 'I don\'t know who you are. I don\'t have money. But what I do have are a very particular set of skills... If you let my daughter go now, that\'ll be the end of it. If you don\'t, I will look for you, I will find you, and I will kill you.';
SET PIN = 'a very particular set of skills';
SET PROMPT = 'fill me in';
SELECT ai_extract($COURSE, {'label': $prompt}):response['label']::varchar AS MY_ANSWER,
CASE WHEN UPPER(MY_ANSWER) = UPPER($PIN) THEN TRUE ELSE FALSE END AS IN_THE_HOLE,
LENGTH($PROMPT) AS STROKES;

### Streams

To help with automation, we'll use a [Snowflake stream](https://docs.snowflake.com/en/user-guide/streams-intro). A stream object records data manipulation language (DML) changes made to tables, including inserts (including COPY INTO), updates, and deletes, as well as metadata about each change, so that actions can be taken using the changed data. 


If you've not worked with streams before, in the following cells we'll walk through an ELI5 example to show how they work. In the following cell we create a simple table and then create a stream that references it to track changes.


In [ ]:
CREATE OR REPLACE TABLE demo_orders (
    order_id     INTEGER,
    customer_id  INTEGER,
    order_amount NUMBER(10,2),
    order_status STRING
);

INSERT INTO demo_orders VALUES
    (1, 101, 250.00, 'NEW'),
    (2, 102, 125.50, 'NEW'),
    (3, 103, 300.00, 'NEW');

CREATE OR REPLACE STREAM demo_orders_stream
ON TABLE demo_orders;

Now that we have a table and stream created, let's make changes to the rows within the table and see how they are recorded in our stream.

In [ ]:

-- Insert a new row
INSERT INTO demo_orders VALUES (4, 104, 175.00, 'NEW');

-- Update an existing row
UPDATE demo_orders
SET order_status = 'SHIPPED'
WHERE order_id = 2;

-- Delete a row
DELETE FROM demo_orders
WHERE order_id = 1;

In [ ]:
SELECT
    order_id,
    customer_id,
    order_amount,
    order_status,
    METADATA$ACTION AS action_type,
    METADATA$ISUPDATE AS is_update
FROM demo_orders_stream;

Now let's use the stream in a hypothetical ETL process and see what happens.

In [ ]:
CREATE OR REPLACE TABLE demo_orders_changes (
    order_id     INTEGER,
    customer_id  INTEGER,
    order_amount NUMBER(10,2),
    order_status STRING,
    action       STRING,
    change_ts    TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

INSERT INTO demo_orders_changes (
    order_id,
    customer_id,
    order_amount,
    order_status,
    action
)
SELECT
    order_id,
    customer_id,
    order_amount,
    order_status,
    METADATA$ACTION
FROM demo_orders_stream;

In [ ]:
SELECT
    order_id,
    customer_id,
    order_amount,
    order_status,
    METADATA$ACTION AS action_type,
    METADATA$ISUPDATE AS is_update
FROM demo_orders_stream;

We can see that our stream has now been "consumed." To hydrate it, run any sort of DML statement on the table 'demo_orders.'

Now that we understand streams, let's build one to track when new PDFs are added to our stage that I very thoughtfully named '@PDF_DUMP.'

The table we'll be tracking changes to is a [directory table](https://docs.snowflake.com/en/user-guide/data-load-dirtables) that exists on the stage by including the syntax `DIRECTORY = ( ENABLE = TRUE )` in the stage creation DDL. 

In [ ]:
CREATE OR REPLACE STREAM PDF_STREAM ON STAGE PDF_DUMP;

We've created our stream, which will represent the start of our pipeline (after a new file has been added to the stage). In the next cell, in a big 'ol mess of SQL, we'll create the final table that will be populated by our process, and the stored procedure that will actually populate it. 

In a more real-life setting, you'd probably try to decompose this wall-of-code Mega-SPROC a bit, but just know it incorporates all of our before-lunch logic as CTEs.  

In [ ]:
CREATE OR REPLACE TABLE PROCESSED_CONTRACTS (
DOC_LOCATION VARCHAR,
DOCUMENT_MARKDOWN VARCHAR, 
VENDOR_NAME VARCHAR,
PAYMENT_TERMS VARCHAR,
CONTRACT_DATE DATE, 
DISCOUNT_LANGUAGE VARCHAR,
LATE_PAYMENT_PENALTY VARCHAR, 
TERMINATION_LANGUAGE VARCHAR, 
NEEDS_HUMAN_REVIEW BOOLEAN,
IDENTIFIED_ISSUES VARCHAR,
SUMMARY VARCHAR
);

CREATE OR REPLACE PROCEDURE PROCESS_CONTRACT_PIPELINE()
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
BEGIN
    INSERT INTO PROCESSED_CONTRACTS (
        DOC_LOCATION,
        DOCUMENT_MARKDOWN,
        VENDOR_NAME,
        PAYMENT_TERMS,
        CONTRACT_DATE,
        DISCOUNT_LANGUAGE,
        LATE_PAYMENT_PENALTY,
        TERMINATION_LANGUAGE,
        NEEDS_HUMAN_REVIEW,
        IDENTIFIED_ISSUES,
        SUMMARY
    )
    WITH PDF_LINKS AS (
        -- Query 1: Get File List
        SELECT RELATIVE_PATH AS DOC_LOCATION,
               TO_FILE('@pdf_dump', RELATIVE_PATH) AS DOC_FILE 
        FROM PDF_STREAM
        WHERE METADATA$ACTION = 'INSERT'
    ),
    PDF_CONTENT AS (
        -- Query 2: Parse Documents
        SELECT DOC_LOCATION, 
               TO_VARCHAR(
                   AI_PARSE_DOCUMENT(
                       DOC_FILE, 
                       {'mode': 'LAYOUT'}
                   )
               ) AS DOCUMENT_MARKDOWN 
        FROM PDF_LINKS
    ),
    DOCUMENT_CLASSIFICATION AS (
        -- Query 3: Classify Document Type
        SELECT *, 
               AI_CLASSIFY(DOCUMENT_MARKDOWN, ['Invoice', 'Contract', 'Other']):labels[0]::VARCHAR AS DOC_TYPE
        FROM PDF_CONTENT
    ),
    CONTRACT_EXTRACTION AS (
        -- Query 4: Extract Data (Filtered for Contracts only)
        SELECT *, 
               AI_EXTRACT(DOCUMENT_MARKDOWN, {
                   'VENDOR_NAME': 'What is the name of the vendor providing the good or service?', 
                   'CONTRACT_DATE': 'What is the date of the contract?',
                   'TERMINATION_LANGUAGE': 'How do the parties involved in this agreement terminate it if necessary.',
                   'LATE_PAYMENT_PENALTY': 'What happens if a payment is not made on-time?',
                   'DISCOUNT_LANGUAGE': 'Describe any discounts available for early payment.'
                   }
               ) AS FIELD_EXTRACTION,
               AI_CLASSIFY(
                 DOCUMENT_MARKDOWN,
                 [
                   {'label': 'NET_10', 'description': 'payment due 10 days after invoice'},
                   {'label': 'NET_15', 'description': 'payment due 15 days after invoice'},
                   {'label': 'NET_20', 'description': 'payment due 20 days after invoice'},
                   {'label': 'NET_21', 'description': 'payment due 20 days after invoice'},
                   {'label': 'NET_30', 'description': 'payment due 30 days after invoice'},
                   {'label': 'NET_45', 'description': 'payment due 45 days after invoice'},
                   {'label': 'NET_60', 'description': 'payment due 60 days after invoice'},
                   {'label': 'NET_90', 'description': 'payment due 90 days after invoice'},
                   {'label': 'DUE_ON_RECEIPT', 'description': 'payment due immediately upon receipt'},
                   {'label': 'ADVANCED_PAYMENT', 'description': 'payment required before goods or services are delivered'},
                   {'label': 'MILESTONE_PAYMENT', 'description': 'payment due upon completion of project milestones'},
                   {'label': 'MONTHLY_IN_ARREARS', 'description': 'payment due on the first or last day of the month for services previously provided'}
                 ],
                 {
                   'task_description': 'Identify the payment terms described in the contract',
                   'output_mode': 'single',
                   'examples': [
                     {
                       'input': 'Invoice payable within ten days of receipt.',
                       'labels': ['NET_10'],
                       'explanation': 'the text specifies payment within ten days'
                     },
                     {
                       'input': 'Prepayment is required at the start of each period.',
                       'labels': ['ADVANCED_PAYMENT'],
                       'explanation': 'payment must be made prior to service delivery'
                     },
                     {
                       'input': 'Payment is due immediately upon receipt of invoice.',
                       'labels': ['DUE_ON_RECEIPT'],
                       'explanation': 'the text states payment is due immediately'
                     },
                     {
                       'input': 'Payments will be made upon completion of each project phase.',
                       'labels': ['MILESTONE_PAYMENT'],
                       'explanation': 'payment is tied to milestones'
                     },
                     {
                       'input': 'Balance payable at the end of each month.',
                       'labels': ['MONTHLY_IN_ARREARS'],
                       'explanation': 'payment is due at month end'
                     }
                   ]
                 }
               ) AS PAYMENT_TERMS_RAW
        FROM DOCUMENT_CLASSIFICATION
        WHERE DOC_TYPE = 'Contract'
    ),
    LLM_JUDGE AS (
        -- Query 5: LLM Judge
        SELECT 
          *,
          AI_COMPLETE(
            model => 'claude-4-sonnet',
            prompt => CONCAT(
            'You are reviewing a contract analysis for accuracy and completeness.

            Review the extracted contract fields and classified payment terms.

            Flag this contract for human review if:
            - Any required field is missing or incorrect
            - The required fields to check for correctness are 
                1. Vendor Name
                2. Termination Language
                3. Late Payment Penalty
            - The payment terms do not match the contract language
            - Payment terms need to only fit a category.
            - The available payment categories are:
                NET_10,
                NET_15, 
                NET_20,
                NET_21, 
                NET_30, 
                NET_45, 
                NET_60, 
                NET_90, 
                DUE_ON_RECEIPT, 
                ADVANCED_PAYMENT,
                MILESTONE_PAYMENT,
                MONTHLY_IN_ARREARS
            - Payment terms need not contain a full explanation of remitting payment, discounts, etc.
            - The vendor does not match the person or company providing the goods or services
            - The contract language is ambiguous or unusual
            - You are not judging output formatting
            - There is conflicting or unclear payment or termination language',
            '<document_text>', DOCUMENT_MARKDOWN, '</document_text>',
            '<extracted_fields>', FIELD_EXTRACTION::VARCHAR, '</extracted_fields>',
            '<payment_terms>', PAYMENT_TERMS_RAW::VARCHAR, '</payment_terms>',

            'Stop and review your work. Make sure you are following instructions exactly.'
            ),
            response_format => {
                'type': 'json',
                'schema': {
                    'type': 'object',
                    'properties': {
                        'needs_human_review': {
                            'type': 'boolean'
                        },
                        'issues': {
                            'type': 'string',
                            'description': 'a list of identified issues (empty if none)'
                        },
                        'summary': {
                            'type': 'string',
                            'description': 'a short explanation (empty if no issues)'
                        }          
                    }
                }
            }
          ) AS LLM_JUDGE_RESULT
        FROM CONTRACT_EXTRACTION
    )
    -- Final Map and Insert
    SELECT 
        DOC_LOCATION,
        DOCUMENT_MARKDOWN,
        FIELD_EXTRACTION['response']['VENDOR_NAME']::VARCHAR AS VENDOR_NAME,
        -- Extract the label from the AI_CLASSIFY result object
        PAYMENT_TERMS_RAW['labels'][0]::VARCHAR AS PAYMENT_TERMS,
        -- Attempt to cast extraction to DATE. 
        TRY_TO_DATE(FIELD_EXTRACTION['response']['CONTRACT_DATE']::VARCHAR, 'MMMM DD, YYYY') AS CONTRACT_DATE,
        FIELD_EXTRACTION['response']['DISCOUNT_LANGUAGE']::VARCHAR AS DISCOUNT_LANGUAGE,
        FIELD_EXTRACTION['response']['LATE_PAYMENT_PENALTY']::VARCHAR AS LATE_PAYMENT_PENALTY,
        FIELD_EXTRACTION['response']['TERMINATION_LANGUAGE']::VARCHAR AS TERMINATION_LANGUAGE,
        LLM_JUDGE_RESULT['needs_human_review']::BOOLEAN AS NEEDS_HUMAN_REVIEW,
        LLM_JUDGE_RESULT['issues']::VARCHAR AS IDENTIFIED_ISSUES,
        LLM_JUDGE_RESULT['summary']::VARCHAR AS SUMMARY
    FROM LLM_JUDGE;

    RETURN 'Contracts processed and inserted successfully.';
END;
$$;

### Tasks
[Tasks](https://docs.snowflake.com/en/user-guide/tasks-intro) are an automation tool in Snowflake. One option is to schedule them to run on an interval, as shown in the commented out line of code below. You could also use cron syntax. Where they get more interesting is that you can create a task triggered by a stream. Additionally, you can create a DAG of task dependencies where one task is triggered by the completion of an upstream task. 

In the cell below, the syntax `WHEN SYSTEM$STREAM_HAS_DATA` will trigger our task when our stream has data. Our stream will have data when we add a PDF to the stage.

In [ ]:
CREATE OR REPLACE TASK PDF_EXTRACTION_TASK
WAREHOUSE = COMPUTE_WH
--SCHEDULE = '30 SECONDS'
WHEN SYSTEM$STREAM_HAS_DATA('PDF_STREAM')
AS CALL PROCESS_CONTRACT_PIPELINE();

ALTER TASK PDF_EXTRACTION_TASK RESUME;

We're now going to grab a few PDFs from the [Github repo](https://github.com/justindelisi-phdata/snowflake_intelligence_hol/tree/minneapolis/cortex_function_lab_pdfs/automation_examples) and manually upload them to a stage and see what happens.

There will be a bit of latency while our stored procedure runs, so to kill some time, enjoy running the cell of AI slop code below. 

In [ ]:
import time
import random
from IPython.display import clear_output, display, Markdown

def humorous_timer(seconds=60):
    # A list of silly "loading" tasks to display at random
    loading_messages = [
        "Reticulating splines...",
        "Convincing the data to cooperate...",
        "Downloading more RAM...",
        "Googling 'how to make time go faster'...",
        "Waking up the hamsters inside the server...",
        "Calculating the last digit of Pi...",
        "Distracting you with this text...",
        "Reversing the polarity of the neutron flow...",
        "Locating the 'Any' key...",
        "Making a coffee. Do you want one?",
    ]

    print("⏱️ Timer started! Don't panic.")
    
    for remaining in range(seconds, -1, -1):
        # Calculate progress for the bar
        progress = seconds - remaining
        bar_length = 20
        filled_length = int(bar_length * progress // seconds)
        bar = '█' * filled_length + '░' * (bar_length - filled_length)
        
        # Determine the "Mood" of the timer based on time left
        if remaining > 45:
            mood = "🟢 Phase 1: Optimism. We have plenty of time."
            current_task = "Initializing..."
        elif remaining > 30:
            mood = "🟡 Phase 2: Boredom. This is taking a while."
            current_task = random.choice(loading_messages)
        elif remaining > 15:
            mood = "🟠 Phase 3: Mild Concern. Did the code freeze?"
            current_task = "Checking pulse..."
        elif remaining > 5:
            mood = "🔴 Phase 4: PANIC. HURRY UP."
            current_task = "SWEATING PROFUSELY."
        elif remaining > 0:
            mood = "🔥 FINAL COUNTDOWN 🔥"
            current_task = "HOLD ON TO YOUR BUTTS."
        else:
            mood = "✨ DONE ✨"
            current_task = "That wasn't so bad, was it?"

        
        # Display the formatted timer
        print(f"## {mood}")
        print(f"### ⏳ {remaining} seconds remaining")
        print(f"[{bar}] {int((progress/seconds)*100)}%")
        print(f"\nSystem Status: {current_task}")
        
        # Wait 1 second
        if remaining > 0:
            time.sleep(1)

    print("\n✅ Timer Complete!")

# Run the timer
humorous_timer(60)

In [ ]:
SELECT * FROM PROCESSED_CONTRACTS;

In [ ]:
ALTER TASK PDF_EXTRACTION_TASK SUSPEND;